# GATraj-RiskMotion on Kaggle

Enable Internet and a GPU accelerator in notebook settings. This notebook retains the original 1,000-epoch training schedule and can run the corrected baseline or proposed motion module. No accuracy improvement is assumed. Train each of the five held-out folds independently; use the same settings and seeds in both arms.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path("/kaggle/working/GATraj-RiskMotion")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/Kedar2002/GATraj-RiskMotion.git", str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before training."
print("GPU:", torch.cuda.get_device_name(0))

## Configure one experiment

Use `MOTION_MODULE=False` for the corrected baseline. Keep `FIX_ATTENTION=True` in both arms. Change component switches for the ablations listed in the README. `TEST_SET`: 0 ETH, 1 Hotel, 2 Zara1, 3 Zara2, 4 University. Outputs for the module combinations and seeds receive distinct names. Keep the full horizon fixed even during a quick startup check; set `STOP_AT_EPOCH=1` to stop early without changing the cosine schedule.

In [ ]:
TEST_SET = 1
SEED = 1
MOTION_MODULE = True
RELATIVE_VELOCITY = True
CLOSEST_APPROACH = True
CONFIDENCE_GATE = True
ADAPTIVE_NEIGHBORS = False
FIX_ATTENTION = True

TOTAL_EPOCHS = 1000
STOP_AT_EPOCH = TOTAL_EPOCHS
EPOCHS_PER_RUN = 100
OUTPUT_DIR = Path("/kaggle/working/runs")

## Train or resume

This cell resumes from the highest numbered checkpoint already present under the selected experiment. To resume across Kaggle runtimes, first attach your saved notebook output as an input dataset and copy its `runs/` directory to `/kaggle/working/runs/`. The script cannot recover files from an unsaved runtime. Save a notebook version with output files to preserve the result.

In [ ]:
from train import get_parser, experiment_name

common_args = [
    "--test_set", str(TEST_SET), "--seed", str(SEED),
    "--num_epochs", str(TOTAL_EPOCHS),
    "--x_encoder_layers", "3", "--eta_min", "1e-5", "--batch_size", "32",
    "--learning_rate", "5e-4", "--randomRotate", "True", "--final_mode", "20",
    "--neighbor_thred", "10", "--using_cuda", "True", "--gpu", "0", "--clip", "1",
    "--pass_time", "2", "--ifGaussian", "False", "--SR", "True", "--input_offset", "True",
    "--fix_attention", str(FIX_ATTENTION), "--motion_module", str(MOTION_MODULE),
    "--relative_velocity", str(RELATIVE_VELOCITY), "--closest_approach", str(CLOSEST_APPROACH),
    "--confidence_gate", str(CONFIDENCE_GATE), "--adaptive_neighbors", str(ADAPTIVE_NEIGHBORS),
    "--save_base_dir", str(OUTPUT_DIR),
]
run_name = experiment_name(get_parser().parse_args(common_args))
common_args += ["--train_model", run_name]
run_dir = OUTPUT_DIR / str(TEST_SET) / run_name

def latest_epoch():
    return max((int(path.stem.rsplit("_", 1)[1])
                for path in run_dir.glob(f"{run_name}_*.tar")), default=0)

assert 0 < STOP_AT_EPOCH <= TOTAL_EPOCHS
assert EPOCHS_PER_RUN > 0
completed = latest_epoch()
print("Experiment:", run_name)
print("Checkpoint directory:", run_dir)
print("Completed epoch:", completed)
while completed < STOP_AT_EPOCH:
    count = min(EPOCHS_PER_RUN, STOP_AT_EPOCH - completed)
    subprocess.run([sys.executable, "train.py", *common_args,
                    "--epochs_per_run", str(count), "--load_model", str(completed)], check=True)
    advanced = latest_epoch()
    if advanced <= completed:
        raise RuntimeError("Training produced no new checkpoint; inspect the output above.")
    completed = advanced
    print("Saved through epoch", completed)
print("Completed through epoch", completed)

## Evaluate the predetermined final checkpoint

The evaluator reports minADE20/minFDE20, observed-density subsets, parameter count, synchronized forward time, and a label-free top-1 collision diagnostic. Use the same GPU for timing comparisons. Early checkpoints are only startup checks; do not pick an epoch by its held-out test error.

In [ ]:
import json

assert latest_epoch() >= TOTAL_EPOCHS, "Finish the predetermined 1,000 epochs before the final comparison."
subprocess.run([sys.executable, "evaluate.py", *common_args,
                "--phase", "test", "--load_model", str(TOTAL_EPOCHS)], check=True)
report = json.loads((run_dir / f"evaluation_{TOTAL_EPOCHS}.json").read_text())
print(json.dumps({key: report[key] for key in ("groups", "parameters", "device", "forward_ms_per_batch")}, indent=2))

## Complete the comparison

Repeat for `TEST_SET=0,1,2,3,4`, then seeds 1, 2 and 3 with baseline and primary model. Average the five fold scores equally within each seed, then report mean and standard deviation across seeds. Keep all training settings fixed and label published paper results separately from reproduced results. See `docs/EXPERIMENTS.md` for the component controls and limitations.

All checkpoints remain in `/kaggle/working/runs`. Plan storage, save notebook outputs, and retain the final model plus any checkpoints needed for resume or analysis.